## Block Interaction analysis

The focus of this notebook is to analyze multi-block operator replacement behaviour, in order to determine propagation error.

The following experiments work with assesment of block replacement interaction in the setting of multiple replaced blocks. The pipeline works with exclusion of first and last blocks due to strong error propagation.

In [ ]:
from dataclasses import asdict
from datetime import datetime, timezone
import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch

In [ ]:
def find_project_root(start):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'mlp_replacement').is_dir():
            return candidate
    raise RuntimeError('Could not locate the repository root')

In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mlp_replacement.capture import collect_modules_io
from mlp_replacement.config import (
    DataConfig,
    ModelConfig,
    OperatorConfig,
    RecoveryConfig,
)
from mlp_replacement.data import build_data_loaders
from mlp_replacement.evaluation.language_model import evaluate_language_model
from mlp_replacement.evaluation.operator import evaluate_operator
from mlp_replacement.model import get_mlp_block, load_model_and_tokenizer
from mlp_replacement.operators import (
    GatedMLPReplacement,
    fit_operator,
    fit_ridge_linear,
)
from mlp_replacement.recovery import cache_teacher_logits, mean_cache_loss
from mlp_replacement.surgery import (
    count_parameters,
    temporary_replacement,
    temporary_replacements,
)

In [ ]:
SEED = 21
torch.manual_seed(SEED)
sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
KL_RECOVERY_UPDATE_BUDGET = 64

In [ ]:
model_config = ModelConfig(
    model_id='HuggingFaceTB/SmolLM2-1.7B',
    device='auto',
    dtype='auto',
)

data_config = DataConfig(
    sequence_length=128,
    batch_size=2,
    num_calibration_batches=48,
    num_operator_validation_batches=24,
    num_recovery_batches=KL_RECOVERY_UPDATE_BUDGET,
    num_recovery_validation_batches=0,
    num_model_validation_batches=24,
    num_test_batches=0,
    seed=SEED,
)

training_config = OperatorConfig(
    epochs=64,
    learning_rate=1e-3,
    batch_size=2048,
    weight_decay=0.0,
    scheduler='constant',
    early_stopping_patience=3,
    seed=SEED,
)

recovery_config = RecoveryConfig(
    enabled=True,
    epochs=1,
    learning_rate=1e-5,
    weight_decay=0.0,
    temperature=1.0,
    cache_dtype='float16',
    early_stopping_patience=None,
)

first and last layers excluded

In [ ]:
model, tokenizer = load_model_and_tokenizer(model_config)
device = next(model.parameters()).device
ELIGIBLE_LAYERS = list(range(1, model.config.num_hidden_layers - 1))

In [ ]:
loaders = build_data_loaders(
    tokenizer,
    data_config,
    include_recovery=True,
)

## Experiments

### Sliding Window Replacement analysis

- tests model degradation with rolling window replacement (e.g. 3 consecutive replacements)
- offset parameter: controls replacement offset (e.g. offset = 1 -> replacement - original - replacement)
- tests:
    - cummulative error across the entire model with multi-block replacement
    - block depth impact (do earlier/latter layers propagate error more?)
    - block offset / replacemnt chaining impact (does spacing between replacement helps to reduce propagated error?)
    - operator architecture placement (where to place linear layers/non-linear layers) based on depth

- ideas:
    - test linear vs non-linear operators

$S(s, k, g) = \{s, s + (g +1), s + 2(g + 1), ...\}$

where:

- $s$ start block index
- $k$ number of blocks to replace
- $g$ offset (skipping between blocks)

example:
- $s = 2, k=3, g=1 \to S = [2, 4, 6]$
- $s = 2, k=3, g=0 \to S = [2, 3, 4]$

- plots:
    - time-series lineplot: x=index, y=value
        - hue/color: linear(orange), swiglu_50(blue)
        - multiplot: first plot k=2, second plot k=3, ... stacked under each other

### Operator Block Interaction

#### Pairwise interaction analysis
- using KL divergence tests:
    - block $i$ using $KL_i$
    - block $j$ using $KL_j$
    - replaced blocks using $K_{ij}$

- tests:
    - $I_{ij} = K_{ij} - (K_i + K_j)$
    - tests error cummulation linearity

- metrics:
    - KL div
    - PPL

- evaluation:
    - if $I_{ij} \approx 0$ -> replacements propagate error additively (idependent?)
    - if $I_{ij} > 0$ -> replacements amplify error propagation
    - if $I_{ij} < 0 $ -> replacement damage is less than expected

- plots:
    - relational matrix heatmap where val=I_ij (full matrix to see ij vs ji)

#### higher order interaction

Use inclusion/exclusion principle for scaling (gpt suggested)



bonus ideas:
- Zeby sa tu dal mozno vyuzit cross quantilogram (CQ) pre testovanie toho impactu medzi blokovo? (to by bolo dost zaujimave takato bakalarka aplikacia). Lag operator by umoznoval sledovat propagaciu erroru maybe, a vizualizovat to cez network graf kruhovy